In [6]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

openai_client = OpenAI(
    api_key=os.getenv("GEMINI_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

try:
    response = openai_client.chat.completions.create(
        model="gemini-2.5-flash",  
        messages=[{"role": "user", "content": "Hello! Confirming that the client works."}]
    )
    print("Success! Response from Gemini:")
    print(response.choices[0].message.content)
except Exception as e:
    print(f"Error: {e}")
    print("Double check your API key and ensure your .env file is in the right directory.")

Success! Response from Gemini:
Hello! That's great news!

To make sure I'm on the same page, could you please specify which client you're referring to? Knowing the context will help me assist you better if you have any follow-up questions.


In [7]:
def llm(prompt):
    response = openai_client.chat.completions.create(
        model="gemini-2.5-flash",
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

In [8]:
question = "I just discovered the course. Can I join now?"
answer = llm(question)
print(answer)

That depends on the specific course you're interested in!

To give you an accurate answer, could you please tell me:

1.  **What is the name of the course?**
2.  **Who is offering the course (e.g., a specific university, an online platform like Coursera/edX/Udemy, a private company, etc.)?**

Different courses have different enrollment models:

*   **Self-paced courses:** Often allow you to join at any time.
*   **Cohort-based courses:** May have specific start dates and enrollment windows, meaning you might need to wait for the next session.
*   **Live courses:** Definitely have fixed start dates and enrollment deadlines.

Once I have more details, I can help you find the specific information you need!


In [9]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

In [10]:
prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

In [11]:
answer = llm(prompt)
print(answer)

Yes, you can join now. However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted.


In [12]:
def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)

In [13]:
import requests
docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

In [14]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1342

In [15]:
from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

In [16]:
search_results=index.search(question,
             filter_dict={'course':'llm-zoomcamp'},
             num_results=5
             )

In [17]:
def search(question, course="llm-zoomcamp"):
    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )

In [18]:
search_results = search(question)

In [1]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

In [2]:
USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

In [19]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

In [22]:
def build_prompt(question,search_results):
    context=build_context(search_results)
    prompt=USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

In [23]:
prompt=build_prompt(question,search_results)

In [24]:
print(prompt)

Question:
I just discovered the course. Can I join now?

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project